In [ ]:
!pip install --quiet polars lets-plot requests geopandas geodatasets

### [Data Visualization with Lets-Plot and Polars](https://medium.com/@shouke.wei/cfe974322fe1)

> Leveraging Native Polars Support for High-Performance Data Visualization with Real-World Datasets

In [ ]:
import pandas as pd
import numpy as np
import polars as pl
import geopandas as gpd
import geodatasets as gd

import requests

from lets_plot import *
from lets_plot.mapping import as_discrete
LetsPlot.setup_html()

from IPython.display import display

import warnings
warnings.filterwarnings('ignore')

co2_url = "https://raw.githubusercontent.com/owid/co2-data/master/owid-co2-data.csv"
pop_url = "https://raw.githubusercontent.com/datasets/population/master/data/population.csv"
stocks_url = "https://raw.githubusercontent.com/plotly/datasets/master/finance-charts-apple.csv"
penguins_url = "https://raw.githubusercontent.com/allisonhorst/palmerpenguins/master/inst/extdata/penguins.csv"
countries_url = "https://raw.githubusercontent.com/dr5hn/countries-states-cities-database/master/csv/countries.csv"
climate_url = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-01-07/temperature.csv"
covid_url = "https://raw.githubusercontent.com/CSSEGISandData/COVID-19/master/csse_covid_19_data/csse_covid_19_daily_reports/03-23-2023.csv"

#### **Basic Scatter Plot: Global CO2 Emissions**

In [ ]:
# Load CO2 emissions data directly from GitHub
# Read with Polars - notice the native integration
co2_df = pl.read_csv(co2_url)
display(co2_df.sample(10))

# Filter for recent years and countries only (exclude regions)
co2_filtered = (
    co2_df
    .filter(pl.col("year") >= 2018)
    .filter(pl.col("iso_code").str.len_chars() == 3)  # Country codes are 3 chars
    .filter(pl.col("co2").is_not_null())
    .filter(pl.col("population").is_not_null())
    .select([
        "country", "year", "co2", "population",
        "gdp", "co2_per_capita"
    ])
)
display(co2_filtered.sample(10))

# Create scatter plot - Lets-Plot works directly with Polars!
p1 = (
    ggplot(co2_filtered, aes(x="gdp", y="co2")) +
    geom_point(aes(color="co2_per_capita", size="population"), alpha=0.7) +
    scale_x_log10() +
    scale_y_log10() +
    scale_color_gradient(low="blue", high="red") +
    labs(
        title="CO2 Emissions vs GDP (2018-2022)",
        subtitle="Size = Population, Color = CO2 per capita",
        x="GDP (log scale)",
        y="CO2 emissions (log scale)"
    ) +
    theme_minimal()
)
p1.show()

#### **Time Series Line Plot: Stock Market Data**

In [ ]:
# Load stock data for major tech companies
# Read and process with Polars
stocks_df = (
    pl.read_csv(stocks_url)
    .with_columns([
        pl.col("Date").str.to_date(),
        pl.col("AAPL.Close").alias("price")
    ])
    .filter(pl.col("Date") >= pl.date(2015, 1, 1))
    .select(["Date", "price"])
    .sort("Date")
    # Add moving averages
    .with_columns([
        pl.col("price").rolling_mean(window_size=30).alias("ma_30"),
        pl.col("price").rolling_mean(window_size=90).alias("ma_90")
    ])
)
display(stocks_df.sample(10))

# Create multi-layer time series plot
p2 = (
    ggplot(stocks_df, aes(x="Date")) +
    geom_line(aes(y="price"), color="black", alpha=0.7) +
    geom_line(aes(y="ma_30"), color="blue", size=1) +
    geom_line(aes(y="ma_90"), color="red", size=1) +
    labs(
        title="Apple Stock Price with Moving Averages",
        subtitle="Black: Daily price, Blue: 30-day MA, Red: 90-day MA",
        x="Date",
        y="Price (USD)"
    ) +
    theme_light()
)
p2.show()

#### **Histogram and Density Plot: Penguins Dataset**

In [ ]:
penguins_df = (
    pl.read_csv(penguins_url)
    .select([
        "species", "island", "bill_length_mm", "bill_depth_mm",
        "flipper_length_mm", "body_mass_g"
    ])
    .drop_nulls()
)
display(penguins_df.sample(10))

# Histogram: bill length by species
p1 = (
    ggplot(penguins_df, aes(x="bill_length_mm", fill="species")) +
    geom_histogram(alpha=0.7, bins=20) +
    facet_wrap("species", ncol=3) +
    labs(
        title="Bill Length Distribution by Penguin Species",
        x="Bill Length (mm)",
        y="Count"
    ) +
    theme_minimal() +
    theme(legend_position="none")
)
p1.show()

# Density plot: flipper length by species
p2 = (
    ggplot(penguins_df, aes(x="flipper_length_mm", color="species")) +
    geom_density(size=1.2, alpha=0.8) +
    labs(
        title="Flipper Length Distribution by Penguin Species",
        x="Flipper Length (mm)",
        y="Density"
    ) +
    theme_minimal()
)
p2.show()

#### **Box Plot: Climate Data Analysis**

In [ ]:
df = pl.read_csv(climate_url, null_values=["NA"])

climate_df = (
    df.filter(pl.col("temp_type") == "max")  # use max temperatures
    .with_columns([
        pl.col("date").str.to_date().alias("date")
    ])
    .filter(pl.col("date") >= pl.date(1950, 1, 1))  # adjusted date filter
    .with_columns([
        pl.col("date").dt.month().alias("month"),
        pl.col("date").dt.year().alias("year")
    ])
    .filter(pl.col("city_name").is_in([
        "CANBERRA", "PERTH", "MELBOURNE", "SYDNEY", "BRISBANE"
    ]))
    .drop_nulls(subset=["temperature"])
    .group_by(["city_name", "year", "month"])
    .agg(pl.col("temperature").mean().alias("avg_temp"))
    .with_columns([
        # Southern Hemisphere seasons
        pl.when(pl.col("month").is_in([12, 1, 2])).then(pl.lit("Summer"))
        .when(pl.col("month").is_in([3, 4, 5])).then(pl.lit("Fall"))
        .when(pl.col("month").is_in([6, 7, 8])).then(pl.lit("Winter"))
        .otherwise(pl.lit("Spring")).alias("season")
    ])
)
display(climate_df.sample(10))

# Box plot: max temperature distribution by city and season
p5 = (
    ggplot(climate_df, aes(x="city_name", y="avg_temp", fill="season")) +
    geom_boxplot(alpha=0.8) +
    coord_flip() +
    labs(
        title="Max Temperature Distribution by City and Season (1950–2020) - Australian Cities",
        x="City",
        y="Average Max Temperature (°C)"
    ) +
    theme_minimal()
)
p5.show()

#### **Heatmap: Correlation Matrix**

In [ ]:
penguins_df = (
    pl.read_csv(penguins_url)
    .select(["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"])
    .drop_nulls()
)
display(penguins_df.sample(10))

stats_list = penguins_df.columns
correlations = []
for i, stat1 in enumerate(stats_list):
    for j, stat2 in enumerate(stats_list):
        corr_value = penguins_df.select([
            pl.corr(stat1, stat2).alias("correlation")
        ]).item()

        correlations.append({
            "var1": stat1,
            "var2": stat2,
            "correlation": corr_value,
            "x_pos": i,
            "y_pos": j
        })
corr_df = pl.DataFrame(correlations)


p6 = (
    ggplot(corr_df, aes(x="var1", y="var2", fill="correlation")) +
    geom_tile() +
    geom_text(aes(label="correlation"), format=".2f", color="white", size=8) +
    scale_fill_gradient2(low="blue", mid="white", high="red", midpoint=0) +
    labs(
        title="Penguins Measurement Correlation Heatmap",
        x="",
        y=""
    ) +
    theme_minimal() +
    theme(axis_text_x=element_text(angle=45))
)

p6.show()

#### **Grouped Bar Chart: Sales Data**

In [ ]:
# Original sales data
sales_data = pl.DataFrame({
    "region": ["North", "South", "East", "West"] * 4,
    "product": ["A", "A", "A", "A", "B", "B", "B", "B",
                "C", "C", "C", "C", "D", "D", "D", "D"],
    "sales": [120, 100, 110, 90, 150, 140, 160, 130,
              80, 95, 85, 75, 200, 180, 220, 190],
    "quarter": ["Q1"] * 16
})
display(sales_data)

# Add variation for multiple quarters and cast sales back to Int64
q2_data = sales_data.with_columns([
    (pl.col("sales") * 1.1).round().cast(pl.Int64).alias("sales"),
    pl.lit("Q2").alias("quarter")
])
q3_data = sales_data.with_columns([
    (pl.col("sales") * 0.95).round().cast(pl.Int64).alias("sales"),
    pl.lit("Q3").alias("quarter")
])
q4_data = sales_data.with_columns([
    (pl.col("sales") * 1.15).round().cast(pl.Int64).alias("sales"),
    pl.lit("Q4").alias("quarter")
])

full_sales = pl.concat([sales_data, q2_data, q3_data, q4_data])

# Create grouped bar chart
p7 = (
    ggplot(full_sales, aes(x="product", y="sales", fill="quarter")) +
    geom_bar(stat="identity", position="dodge", alpha=0.8) +
    facet_wrap("region", ncol=2) +
    labs(
        title="Sales Performance by Product, Region, and Quarter",
        x="Product",
        y="Sales ($000s)"
    ) +
    theme_minimal()
)
p7.show()

#### **Advanced: Violin Plot with Overlays**

In [ ]:
penguins_df = (
    pl.read_csv(penguins_url)
    .select(["species", "body_mass_g"])
    .drop_nulls()
)

p8 = (
    ggplot(penguins_df, aes(x="species", y="body_mass_g")) +
    geom_violin(aes(fill="species"), alpha=0.7) +
    geom_boxplot(width=0.2, alpha=0.8, outlier_alpha=0.3) +
    stat_summary(fun="mean", geom="point", color="red", size=3) +
    labs(
        title="Body Mass Distribution by Penguin Species",
        subtitle="Violin plot with boxplot overlay and mean points",
        x="Species",
        y="Body Mass (g)"
    ) +
    theme_minimal() +
    theme(
        axis_text_x=element_text(angle=45),
        legend_position="none"
    )
)

p8.show()

#### **Geographic Visualization: World Map**

In [ ]:
# Load real world countries data from GitHub using Polars
# Read the CSV data with Polars
countries_df = pl.read_csv(countries_url)

# Clean and prepare data for mapping
# Select relevant columns and handle missing values
map_data = countries_df.select([
    "name",
    "iso2",
    "iso3",
    "latitude",
    "longitude",
    "region",
    "subregion",
    "capital",
    "currency_name"
]).filter(
    # Remove rows with missing coordinates
    (pl.col("latitude").is_not_null()) &
    (pl.col("longitude").is_not_null())
)

# Create a world map showing countries by region
world_map = (
    ggplot(map_data, aes("longitude", "latitude")) +
    geom_point(aes(color="region", size="latitude"), alpha=0.7) +
    scale_size(range=[2, 8]) +
    labs(
        title="World Countries by Region",
        subtitle="Geographic distribution colored by continental region",
        x="Longitude",
        y="Latitude",
        color="Region",
        caption="Data source: GitHub - countries-states-cities-database"
    ) +
    theme_minimal() +
    ggsize(900, 600)
)

world_map.show()

#### **Interactive Features and Customization**

In [ ]:
# Load real world countries data from GitHub using Polars
# Read the CSV data with Polars
countries_df = pl.read_csv(countries_url)

# Clean and prepare data for mapping
# Select relevant columns and handle missing values
map_data = countries_df.select([
    "name",
    "iso2",
    "iso3",
    "latitude",
    "longitude",
    "region",
    "subregion",
    "capital",
    "currency_name"
]).filter(
    # Remove rows with missing coordinates
    (pl.col("latitude").is_not_null()) &
    (pl.col("longitude").is_not_null())
)

# Advanced customization example
custom_theme = (
    theme_minimal() +
    theme(
        plot_title=element_text(size=16, face="bold"),
        plot_subtitle=element_text(size=12, color="gray"),
        axis_text=element_text(size=10),
        legend_position="bottom",
        panel_grid_minor=element_blank()
    )
)

# Create a world map showing countries by region with advanced customization
world_map = (
    ggplot(map_data, aes(x="longitude", y="latitude", color="region")) +
    geom_point(aes(size="latitude"), alpha=0.7) +
    scale_size(range=[2, 8]) +
    labs(
        title="World Countries Geographic Distribution",
        subtitle="Size represents latitude, colored by region",
        x="Longitude",
        y="Latitude",
        color="Region",
        size="Latitude"
    ) +
    custom_theme
)

world_map.show()

#### **Performance Tips and Best Practices**

In [ ]:
# Load CO2 emissions data from Our World in Data GitHub repository
# Read the CSV data with Polars
co2_df = pl.read_csv(co2_url)

# Efficient data processing with Polars before plotting
efficient_workflow_example = (
    co2_df
    .lazy()  # Use lazy evaluation
    .filter(pl.col("year") >= 2020)
    .filter(pl.col("iso_code").str.len_chars() == 3)
    .filter(pl.col("co2").is_not_null())
    .filter(pl.col("population").is_not_null())
    .filter(pl.col("population") > 0)  # Avoid division by zero
    .with_columns([
        pl.col("co2").cast(pl.Float64),
        pl.col("population").cast(pl.Float64)
    ])
    .select([
        "country", "year", "co2", "population",
        (pl.col("co2") / pl.col("population") * 1000).alias("co2_per_1000")
    ])
    .group_by("country")
    .agg([
        pl.col("co2").mean().alias("avg_co2"),
        pl.col("co2_per_1000").mean().alias("avg_co2_per_1000"),
        pl.col("population").mean().alias("avg_population")
    ])
    .filter(pl.col("avg_co2").is_not_null())  # Remove null results
    .sort("avg_co2", descending=True)
    .head(20)
    .collect()  # Execute lazy operations
)

# Plot the processed data
p10 = (
    ggplot(efficient_workflow_example, aes(x="avg_co2_per_1000", y="avg_co2")) +
    geom_point(aes(size="avg_population", color="country"), alpha=0.7) +
    geom_text(aes(label="country"), hjust=-0.1, size=8) +
    scale_x_log10() +
    scale_y_log10() +
    labs(
        title="Top 20 CO2 Emitters: Total vs Per Capita (2020+)",
        x="Average CO2 per 1000 people (log scale)",
        y="Average Total CO2 Emissions (log scale)"
    ) +
    theme_minimal()
)
p10.show()

### [Beautiful, Interactive Maps with Lets-Plot and GeoPandas](https://medium.com/@shouke.wei/130be1f35af6)

> Interactive maps with pandas

#### **First Map in 30 Seconds**

In [ ]:
chicago = gpd.read_file(gd.get_path("geoda.chicago_commpop"))
display(chicago.head())

In [ ]:
(
    ggplot()
    + geom_map(aes(fill='POP2010'), data=chicago, size=.3)
    + scale_fill_viridis(option='plasma', name='Population')
    + ggtitle('Chicago Population Density (2010)')
    + theme_minimal()
)

#### **Coordinate Reference Systems (CRS)**

In [ ]:
gdf_utm = chicago.to_crs("EPSG:32616")       # UTM zone 16N
ggplot() + geom_map(map=gdf_utm, use_crs="EPSG:32616")

#### **Interactive Live Maps**

In [ ]:
(
    ggplot()
    + geom_livemap(zoom=10)                      # basemap
    + geom_map(aes(fill='POP2010'), data=chicago, alpha=.7)
    + scale_fill_gradient(low="#ffffcc", high="#800026")
)

#### **Layers: Points on Polygons**

In [ ]:
stores = gpd.read_file(gd.get_path("geoda.groceries"))
stores = stores.to_crs(chicago.crs)            # match CRS

(
    ggplot()
    + geom_map(aes(fill='POP2010'), data=chicago, alpha=.4)
    + geom_point(aes(color='Category'), data=stores, size=3)
    + scale_color_brewer(type='qualitative', palette='Set2')
    + ggtitle('Grocery Stores vs. Population')
)

#### **Bubble Maps & Scaling**

In [ ]:
stores['annual_sales'] = stores['YEAR'] * 1_000   # dummy data
(
    ggplot()
    + geom_livemap()
    + geom_point(aes(size='annual_sales', color='STORE_TYPE'),
                 data=stores, alpha=.8, tooltips=layer_tooltips()
                    .line('@STORE_TYPE')
                    .line('Sales: @annual_sales'))
)

#### **Faceting & Small Multiples**

In [ ]:
chicago['POP1990'] = chicago['POP2010'] * 0.9   # dummy
chicago['POP2000'] = chicago['POP2010'] * 0.95

chicago_melt = chicago[['community','geometry','POP1990','POP2000','POP2010']].melt(
    id_vars=['community','geometry'], var_name='year', value_name='pop')
(
    ggplot(chicago_melt)
    + geom_map(aes(fill='pop'), size=.2)
    + facet_wrap('year')
    + scale_fill_viridis()
    + coord_fixed()
)

#### **Themes & Publishing**

```python
+ theme_classic()      # ggplot2 look
+ theme_minimal2()     # modern minimal
+ theme_void()         # no axes, perfect for pure maps

fig = (ggplot() + ...)
ggsave(fig, 'my_map.png', dpi=300, width=16, height=9)
```

#### **Airbnb Prices in Boston**

In [ ]:
boston_hoods = gpd.read_file(gd.get_path("geoda.boston_tracts"))
listings = pd.read_csv('airbnb_boston.csv')   # contains lat, lon, price
listings = gpd.GeoDataFrame(
    listings,
    geometry=gpd.points_from_xy(listings.longitude, listings.latitude),
    crs="EPSG:4326"
)

tracts = boston_hoods.to_crs("EPSG:4326")
listings['tract_id'] = gpd.sjoin(listings, tracts[['geometry','GEOID']])['GEOID']
price_by_tract = listings.groupby('tract_id').price.median().reset_index()
tracts = tracts.merge(price_by_tract, left_on='GEOID', right_on='tract_id')

prices_fig = (
    ggplot()
    + geom_livemap()
    + geom_map(aes(fill='price'), data=tracts,
               tooltips=layer_tooltips().line('Median price: @price'))
    + scale_fill_gradient(low='#edf8e9', high='#005a32', name='USD / night')
    + ggtitle('Median Airbnb Price by Census Tract, Boston')
    + theme_void()
)

### [Data Visualization Mastery with Lets-Plot and Pandas](https://medium.com/@shouke.wei/2fd18535fd1a)

> Unleashing the Power of Grammar of Graphics for Real-World Data Analysis

#### **Scatter Plots with COVID-19 Data**

In [ ]:
# Load COVID-19 data from Johns Hopkins GitHub repository
covid_data = pd.read_csv(covid_url)

# Clean and prepare data
covid_clean = covid_data.dropna(subset=['Confirmed', 'Deaths'])
covid_clean = covid_clean[covid_clean['Confirmed'] > 1000]  # Filter for significant cases
# Create scatter plot
scatter_plot = (ggplot(covid_clean, aes(x='Confirmed', y='Deaths')) +
                geom_point(aes(color='Country_Region'), alpha=0.7, size=2) +
                scale_x_log10() +
                scale_y_log10() +
                labs(
                    title="COVID-19: Confirmed Cases vs Deaths (Log Scale)",
                    subtitle="Data from Johns Hopkins University",
                    x="Confirmed Cases (log scale)",
                    y="Deaths (log scale)",
                    color="Country/Region"
                ) +
                theme_minimal() +
                theme(legend_position='none'))  # Too many countries for legend
display(scatter_plot)

#### **Line Charts with Stock Market Data**

In [ ]:
# Function to fetch stock data (simplified version)
def get_stock_data(symbol, start_date, end_date):
    """Fetch stock data - in practice, use yfinance or similar library"""
    # This is a placeholder - replace with actual data fetching
    dates = pd.date_range(start=start_date, end=end_date, freq='D')
    np.random.seed(42)
    prices = 100 + np.cumsum(np.random.randn(len(dates)) * 0.5)
    return pd.DataFrame({'Date': dates, 'Price': prices, 'Symbol': symbol})

# Create sample stock data for multiple companies
stocks = ['AAPL', 'GOOGL', 'MSFT', 'TSLA']
stock_data = pd.concat([
    get_stock_data(stock, '2023-01-01', '2023-12-31')
    for stock in stocks
], ignore_index=True)
# Create line plot
line_plot = (ggplot(stock_data, aes(x='Date', y='Price', color='Symbol')) +
             geom_line(size=1.2) +
             scale_color_brewer(type='qual', palette='Set2') +
             labs(
                 title="Stock Price Trends Throughout 2023",
                 subtitle="Simulated data for demonstration",
                 x="Date",
                 y="Stock Price ($)",
                 color="Company"
             ) +
             theme_classic() +
             theme(axis_text_x=element_text(angle=45)))
display(line_plot)

#### **Histograms and Density Plots with World Population Data**

In [ ]:
# Load world population data
pop_data = pd.read_csv(pop_url)

# Filter for recent year and clean data
pop_2020 = pop_data[pop_data['Year'] == 2020].copy()
pop_2020 = pop_2020[pop_2020['Value'] > 100000]  # Countries with >100k population
# Log transform for better visualization
pop_2020['Log_Population'] = np.log10(pop_2020['Value'])
# Create histogram
histogram_plot = (ggplot(pop_2020, aes(x='Log_Population')) +
                  geom_histogram(bins=30, fill='steelblue', alpha=0.7, color='white') +
                  geom_density(color='red', size=1) +
                  labs(
                      title="Distribution of World Population by Country (2020)",
                      subtitle="Log-transformed population values",
                      x="Population (log10 scale)",
                      y="Frequency"
                  ) +
                  theme_minimal())
display(histogram_plot)

#### **Box Plots with Climate Data**

In [ ]:
# Simulate climate data (replace with real NOAA data in practice)
np.random.seed(123)
months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
          'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
cities = ['New York', 'Los Angeles', 'Chicago', 'Houston', 'Phoenix']

climate_data = []
for city in cities:
    for month in months:
        # Simulate temperature data with seasonal patterns
        base_temp = 60 + 20 * np.sin(2 * np.pi * months.index(month) / 12)
        if city == 'Phoenix':
            base_temp += 15
        elif city == 'Chicago':
            base_temp -= 10

        temps = np.random.normal(base_temp, 5, 30)  # 30 days of data
        for temp in temps:
            climate_data.append({
                'City': city,
                'Month': month,
                'Temperature': temp
            })
climate_df = pd.DataFrame(climate_data)
# Create box plot
box_plot = (ggplot(climate_df, aes(x='Month', y='Temperature', fill='City')) +
            geom_boxplot(alpha=0.8) +
            scale_fill_brewer(type='qual', palette='Set3') +
            labs(
                title="Monthly Temperature Distribution by City",
                subtitle="Simulated climate data",
                x="Month",
                y="Temperature (°F)",
                fill="City"
            ) +
            theme_light() +
            theme(axis_text_x=element_text(angle=45)))
display(box_plot)

#### **Heatmaps with Correlation Data**

In [ ]:
# Load economic indicators data (simulated)
np.random.seed(456)
countries = ['USA', 'China', 'Japan', 'Germany', 'UK', 'France', 'India', 'Italy']
indicators = ['GDP_Growth', 'Inflation', 'Unemployment', 'Trade_Balance', 'Debt_to_GDP']

# Create correlation matrix
econ_data = pd.DataFrame(
    np.random.randn(len(countries), len(indicators)),
    index=countries,
    columns=indicators
)
# Calculate correlation matrix
corr_matrix = econ_data.corr()
# Reshape for plotting
corr_melted = corr_matrix.reset_index().melt(id_vars='index')
corr_melted.columns = ['Var1', 'Var2', 'Correlation']

# Create heatmap
heatmap_plot = (ggplot(corr_melted, aes(x='Var1', y='Var2', fill='Correlation')) +
                geom_tile() +
                scale_fill_gradient2(low='red', mid='white', high='blue', midpoint=0) +
                labs(
                    title="Correlation Matrix of Economic Indicators",
                    subtitle="Simulated economic data",
                    x="Economic Indicator",
                    y="Economic Indicator",
                    fill="Correlation"
                ) +
                theme_minimal() +
                theme(axis_text_x=element_text(angle=45)))
display(heatmap_plot)

#### **Faceted Plots**

In [ ]:
# Using our climate data from earlier
faceted_plot = (ggplot(climate_df, aes(x='Month', y='Temperature')) +
                geom_boxplot(aes(fill='Month'), alpha=0.7) +
                facet_wrap('City', scales='free_y') +
                scale_fill_hue() +
                labs(
                    title="Temperature Distribution by Month and City",
                    subtitle="Faceted view showing seasonal patterns",
                    x="Month",
                    y="Temperature (°F)"
                ) +
                theme_minimal() +
                theme(
                    axis_text_x=element_text(angle=90, size=8),
                    legend_position='none'
                ))
display(faceted_plot)

#### **Combined Plot Types**

In [ ]:
# Create a comprehensive COVID-19 dashboard data
# (In practice, load from multiple real sources)
np.random.seed(789)
dates = pd.date_range('2020-03-01', '2023-03-01', freq='D')
countries = ['USA', 'India', 'Brazil', 'Russia', 'France']

covid_time_series = []
for country in countries:
    base_cases = np.random.randint(1000, 10000)
    cases = []
    deaths = []

    for i, date in enumerate(dates):
        # Simulate waves
        wave_factor = 1 + 0.5 * np.sin(2 * np.pi * i / 180)  # ~6 month cycles
        daily_cases = int(base_cases * wave_factor * np.random.uniform(0.8, 1.2))
        daily_deaths = int(daily_cases * 0.02 * np.random.uniform(0.5, 1.5))  # ~2% mortality

        cases.append(daily_cases)
        deaths.append(daily_deaths)

        covid_time_series.append({
            'Date': date,
            'Country': country,
            'Daily_Cases': daily_cases,
            'Daily_Deaths': daily_deaths,
            'Cases_7day_avg': np.mean(cases[-7:]) if len(cases) >= 7 else daily_cases
        })
covid_ts_df = pd.DataFrame(covid_time_series)
# Create multi-layered plot
combined_plot = (ggplot(covid_ts_df, aes(x='Date', y='Cases_7day_avg', color='Country')) +
                 geom_line(size=1.2, alpha=0.8) +
                 geom_smooth(method='loess', se=False, size=0.8) +
                 scale_y_log10() +
                 scale_color_brewer(type='qual', palette='Dark2') +
                 labs(
                     title="COVID-19 Daily Cases Trend (7-day Average)",
                     subtitle="With LOESS smoothing trend lines",
                     x="Date",
                     y="Daily Cases (log scale)",
                     color="Country"
                 ) +
                 theme_economist() +
                 theme(
                     legend_position='right',
                     plot_title=element_text(size=14, face='bold')
                 ))
display(combined_plot)

#### **Custom Themes and Colors**

In [ ]:
# Define custom theme
custom_theme = (theme_minimal() +
                theme(
                    plot_background=element_rect(fill='#f8f9fa'),
                    panel_grid_major=element_line(color='#dee2e6', size=0.5),
                    panel_grid_minor=element_blank(),
                    text=element_text(family='Arial', color='#212529'),
                    plot_title=element_text(size=16, face='bold', hjust=0.5),
                    plot_subtitle=element_text(size=12, hjust=0.5, color='#6c757d'),
                    axis_title=element_text(size=11, face='bold'),
                    legend_background=element_rect(fill='white', color='#dee2e6')
                ))

#### **Interactive Elements and Annotations**

In [ ]:
# Create an annotated scatter plot
annotated_plot = (ggplot(covid_clean.head(50), aes(x='Confirmed', y='Deaths')) +
                  geom_point(aes(size='Confirmed'), alpha=0.6, color='steelblue') +
                  geom_text(aes(label='Country_Region'),
                           size=8, nudge_y=0.1, check_overlap=True) +
                  scale_x_log10() +
                  scale_y_log10() +
                  scale_size_continuous(guide='none') +
                  labs(
                      title="COVID-19 Cases vs Deaths with Country Labels",
                      subtitle="Top 50 countries by confirmed cases",
                      x="Confirmed Cases (log scale)",
                      y="Deaths (log scale)"
                  ) +
                  theme_classic())

display(annotated_plot)

# Apply custom theme to a plot
styled_plot = (ggplot(pop_2020.head(20), aes(x='Country_Name', y='Value')) +
               geom_col(fill='#007bff', alpha=0.8) +
               coord_flip() +
               scale_y_continuous(labels=lambda x: f'{x/1e6:.1f}M') +
               labs(
                   title="Top 20 Countries by Population (2020)",
                   subtitle="Data from World Bank",
                   x="Country",
                   y="Population (Millions)"
               ) +
               custom_theme)

display(styled_plot)

#### **Regression Analysis Plots**

In [ ]:
# Create regression plot with confidence intervals
regression_data = covid_clean[covid_clean['Confirmed'] > 10000].copy()
regression_data['Death_Rate'] = regression_data['Deaths'] / regression_data['Confirmed']

regression_plot = (ggplot(regression_data, aes(x='Confirmed', y='Deaths')) +
                   geom_point(alpha=0.6, color='darkred') +
                   geom_smooth(method='lm', se=True, color='blue') +
                   scale_x_log10() +
                   scale_y_log10() +
                   labs(
                       title="Linear Relationship: COVID-19 Cases vs Deaths",
                       subtitle="With 95% confidence interval",
                       x="Confirmed Cases (log scale)",
                       y="Deaths (log scale)"
                   ) +
                   theme_bw())

display(regression_plot)

#### **Distribution Comparisons**

In [ ]:
# Compare distributions across groups
comparison_plot = (ggplot(climate_df, aes(x='Temperature', fill='City')) +
                   geom_density(alpha=0.7) +
                   facet_wrap('City', scales='free_y') +
                   scale_fill_brewer(type='qual', palette='Set2') +
                   labs(
                       title="Temperature Distribution Comparison by City",
                       subtitle="Density plots showing distribution shapes",
                       x="Temperature (°F)",
                       y="Density"
                   ) +
                   theme_light() +
                   theme(legend_position='none'))

display(comparison_plot)

#### **Data Aggregation and Sampling**

In [ ]:
# Working with large datasets - aggregation example
def aggregate_large_dataset(df, group_cols, agg_dict):
    """Aggregate large dataset for visualization"""
    return df.groupby(group_cols).agg(agg_dict).reset_index()

# Example with time series data
monthly_covid = covid_ts_df.groupby(['Country', pd.Grouper(key='Date', freq='M')]).agg({
    'Daily_Cases': 'mean',
    'Daily_Deaths': 'mean'
}).reset_index()
monthly_plot = (ggplot(monthly_covid, aes(x='Date', y='Daily_Cases', color='Country')) +
                geom_line(size=1) +
                geom_point(size=2) +
                scale_y_log10() +
                labs(
                    title="Monthly Average COVID-19 Cases",
                    subtitle="Aggregated from daily data",
                    x="Month",
                    y="Average Daily Cases (log scale)",
                    color="Country"
                ) +
                theme_minimal())
display(monthly_plot)

#### **Performance Optimization Tips**

In [ ]:
# Performance optimization techniques
def optimize_plot_data(df, max_points=10000):
    """Optimize dataframe for plotting"""
    if len(df) > max_points:
        # Sample data while preserving distribution
        return df.sample(n=max_points, random_state=42)
    return df

# Example usage
optimized_data = optimize_plot_data(covid_ts_df)
print(f"Reduced data from {len(covid_ts_df)} to {len(optimized_data)} points")

#### **Color and Accessibility**

In [ ]:
# Colorblind-friendly palette example
colorblind_safe_plot = (ggplot(climate_df[climate_df['City'].isin(['New York', 'Los Angeles', 'Chicago'])],
                               aes(x='Month', y='Temperature', color='City')) +
                        geom_boxplot() +
                        scale_color_manual(values=['#1f77b4', '#ff7f0e', '#2ca02c']) +  # Colorblind safe
                        labs(
                            title="Temperature Comparison (Colorblind-Safe Palette)",
                            x="Month",
                            y="Temperature (°F)",
                            color="City"
                        ) +
                        theme_minimal())

display(colorblind_safe_plot)

#### **Data Integrity and Validation**

In [ ]:
# Data validation before plotting
def validate_plot_data(df, required_cols, numeric_cols):
    """Validate data before creating plots"""
    # Check required columns
    missing_cols = set(required_cols) - set(df.columns)
    if missing_cols:
        raise ValueError(f"Missing required columns: {missing_cols}")

    # Check for null values in critical columns
    null_counts = df[required_cols].isnull().sum()
    if null_counts.any():
        print(f"Warning: Null values found:\n{null_counts[null_counts > 0]}")

    # Check numeric columns
    for col in numeric_cols:
        if not pd.api.types.is_numeric_dtype(df[col]):
            print(f"Warning: {col} is not numeric")

    return True

# Example usage
validate_plot_data(covid_clean, ['Confirmed', 'Deaths', 'Country_Region'], ['Confirmed', 'Deaths'])

#### **Pandas Integration**

In [ ]:
# Seamless integration with pandas operations
def create_analysis_pipeline(data_source, plot_type='scatter'):
    """Complete analysis pipeline with visualization"""

    # Data loading and cleaning
    df = pd.read_csv(data_source)
    df_clean = df.dropna()

    # Exploratory analysis
    summary_stats = df_clean.describe()

    # Visualization based on type
    if plot_type == 'scatter':
        plot = (ggplot(df_clean, aes(x=df_clean.columns[0], y=df_clean.columns[1])) +
                geom_point(alpha=0.6) +
                theme_minimal())

    return {
        'data': df_clean,
        'summary': summary_stats,
        'plot': plot
    }

#### **Export and Sharing**

In [ ]:
# Export plots in different formats
def export_plot(plot, filename, width=10, height=6, dpi=300):
    """Export plot to various formats"""
    try:
        # Save as PNG
        ggsave(plot, filename + '.png', width=width, height=height, dpi=dpi)

        # Save as SVG for vector graphics
        ggsave(plot, filename + '.svg', width=width, height=height)

        print(f"Plot exported as {filename}.png and {filename}.svg")
    except Exception as e:
        print(f"Export failed: {e}")
# export_plot(scatter_plot, 'covid_analysis')

#### **Animation Concepts**

In [ ]:
# Prepare data for animation (conceptual - actual animation requires additional setup)
def prepare_animation_data(df, time_col, value_col, group_col):
    """Prepare data for animated plots"""
    animation_frames = []

    for time_point in df[time_col].unique():
        frame_data = df[df[time_col] == time_point].copy()
        frame_data['frame'] = time_point
        animation_frames.append(frame_data)

    return pd.concat(animation_frames, ignore_index=True)

# Example preparation
anim_data = prepare_animation_data(covid_ts_df, 'Date', 'Daily_Cases', 'Country')

#### **Complex Multi-Plot Layouts**

In [ ]:
# Create subplot layout (conceptual approach)
def create_dashboard_layout(data_dict):
    """Create multi-plot dashboard layout"""
    plots = {}

    # Plot 1: Time series
    plots['timeseries'] = (ggplot(data_dict['time_data'], aes(x='Date', y='Value')) +
                          geom_line() +
                          theme_minimal())

    # Plot 2: Distribution
    plots['distribution'] = (ggplot(data_dict['dist_data'], aes(x='Value')) +
                            geom_histogram() +
                            theme_minimal())

    # Plot 3: Correlation
    plots['correlation'] = (ggplot(data_dict['corr_data'], aes(x='X', y='Y')) +
                           geom_point() +
                           theme_minimal())

    return plots